[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/07_linear_least_squares/first_principles.ipynb)

# Topic 07: Linear Least Squares

## 1. First-Principles Intuition & Motivation

Take $m$ measurements to determine $n \lt m$ parameters. Each measurement contributes one linear equation, so you obtain

$$
A\mathbf{x} = \mathbf{b}, \qquad A \in \mathbb{R}^{m \times n}, \quad \mathbf{b} \in \mathbb{R}^{m}, \quad m \gt n .
$$

The columns of $A$ span a subspace $\mathcal{R}(A) \subseteq \mathbb{R}^{m}$ of dimension at most $n$. Because $n \lt m$, that subspace is a thin sheet inside a large space, and a noisy measurement vector $\mathbf{b}$ lands off the sheet with probability one. **The system has no solution — not because the model is wrong, but because there is more data than freedom.**

The engineering response is to stop demanding equality and start demanding *closeness*. Define the residual $\mathbf{r}(\mathbf{x}) = \mathbf{b} - A\mathbf{x}$ and minimize a norm of it. Different norms give different problems: the $1$-norm gives robust (median-like) regression, the $\infty$-norm gives Chebyshev fitting, and both require linear programming. The $2$-norm is special.

### Why the 2-norm makes the problem linear

Minimizing $\phi(\mathbf{x}) = \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2$ is minimizing a *quadratic*: its gradient is affine, so the stationarity condition is a **linear system**. Three independent stories all converge on the same equations.

1. **Calculus.** $\nabla \phi(\mathbf{x}) = 2A^{\top}(A\mathbf{x} - \mathbf{b})$; setting it to zero gives $A^{\top}A\mathbf{x} = A^{\top}\mathbf{b}$. The Hessian $2A^{\top}A$ is positive semidefinite, so every stationary point is a global minimum.
2. **Geometry.** The closest point of a subspace to $\mathbf{b}$ is its orthogonal projection; the error vector must be perpendicular to the subspace, i.e. $A^{\top}\mathbf{r} = \mathbf{0}$ — the same equations, obtained without a single derivative.
3. **Statistics.** If $\mathbf{b} = A\mathbf{x}_{\text{true}} + \boldsymbol{\varepsilon}$ with $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 I)$, the log-likelihood is $-\frac{1}{2\sigma^2}\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 + \text{const}$, so least squares *is* maximum likelihood under i.i.d. Gaussian noise, and by Gauss–Markov it is the minimum-variance unbiased linear estimator.

That triple coincidence is why least squares, not $\ell_1$ or $\ell_\infty$ fitting, is the workhorse of science. The numerical difficulty appears only when we ask *how* to solve $A^{\top}A\mathbf{x} = A^{\top}\mathbf{b}$ in finite precision — and the answer is: usually, don't.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Linear least-squares problem).** Given $A \in \mathbb{R}^{m \times n}$ and $\mathbf{b} \in \mathbb{R}^{m}$, find

$$
\hat{\mathbf{x}} \in \arg\min_{\mathbf{x} \in \mathbb{R}^{n}} \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 .
$$

The vector $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$ is the *residual*, and $\rho = \Vert \mathbf{r} \Vert_2$ the *residual norm*.

**Definition 2 (Orthogonal projector).** $P \in \mathbb{R}^{m \times m}$ is an orthogonal projector onto a subspace $S$ if $P^2 = P$, $P^{\top} = P$, and $\mathcal{R}(P) = S$. For full-column-rank $A$ the projector onto $\mathcal{R}(A)$ is $P = A(A^{\top}A)^{-1}A^{\top}$; if $A = QR$ is a thin QR factorization, $P = QQ^{\top}$.

**Definition 3 (Reduced SVD and pseudoinverse).** With $\operatorname{rank}(A) = r$ write $A = U_r \Sigma_r V_r^{\top}$, where $U_r \in \mathbb{R}^{m \times r}$ and $V_r \in \mathbb{R}^{n \times r}$ have orthonormal columns and $\Sigma_r = \operatorname{diag}(\sigma_1 \ge \cdots \ge \sigma_r \gt 0)$. The Moore–Penrose pseudoinverse is

$$
A^{+} = V_r \Sigma_r^{-1} U_r^{\top} = \sum_{i=1}^{r} \frac{1}{\sigma_i} \mathbf{v}_i \mathbf{u}_i^{\top} .
$$

**Definition 4 (2-norm condition number, rectangular case).** For $A$ of full column rank,

$$
\kappa_2(A) = \frac{\sigma_1(A)}{\sigma_n(A)} = \Vert A \Vert_2 \, \Vert A^{+} \Vert_2 .
$$

**Definition 5 (Thin QR factorization).** $A = QR$ with $Q \in \mathbb{R}^{m \times n}$ satisfying $Q^{\top}Q = I_n$ and $R \in \mathbb{R}^{n \times n}$ upper triangular. If $A$ has full column rank, $R$ is nonsingular and the factorization is unique up to signs of the rows of $R$; the convention $r_{ii} \gt 0$ makes it unique.

**Definition 6 (Householder reflector).** For $\mathbf{v} \neq \mathbf{0}$, $H = I - 2\dfrac{\mathbf{v}\mathbf{v}^{\top}}{\mathbf{v}^{\top}\mathbf{v}}$ is symmetric and orthogonal ($H^{\top} = H$, $H^2 = I$). Choosing $\mathbf{v} = \mathbf{a} - \alpha \mathbf{e}_1$ with $\alpha = -\operatorname{sign}(a_1)\Vert \mathbf{a} \Vert_2$ gives $H\mathbf{a} = \alpha \mathbf{e}_1$, annihilating an entire column below the diagonal in one orthogonal step.

**Definition 7 (Tikhonov / ridge problem).** For $\lambda \gt 0$,

$$
\hat{\mathbf{x}}_\lambda = \arg\min_{\mathbf{x}} \left( \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 + \lambda \Vert \mathbf{x} \Vert_2^2 \right) .
$$

**Theorem 1 (Normal equations; existence).** $\hat{\mathbf{x}}$ minimizes $\Vert A\mathbf{x} - \mathbf{b} \Vert_2$ if and only if

$$
A^{\top}A\hat{\mathbf{x}} = A^{\top}\mathbf{b},
$$

equivalently $A^{\top}\mathbf{r} = \mathbf{0}$, i.e. $\mathbf{r} \perp \mathcal{R}(A)$. A minimizer always exists because $A^{\top}\mathbf{b} \in \mathcal{R}(A^{\top}) = \mathcal{R}(A^{\top}A)$.

**Theorem 2 (Uniqueness).** The minimizer is unique if and only if $\operatorname{rank}(A) = n$, in which case $A^{\top}A$ is symmetric positive definite and $\hat{\mathbf{x}} = (A^{\top}A)^{-1}A^{\top}\mathbf{b} = A^{+}\mathbf{b}$. If $\operatorname{rank}(A) = r \lt n$, the solution set is the affine subspace $\hat{\mathbf{x}}_{\min} + \mathcal{N}(A)$ of dimension $n - r$.

**Theorem 3 (Pythagoras / minimum-norm solution).** For any $\mathbf{x}$, $\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert A(\mathbf{x} - \hat{\mathbf{x}}) \Vert_2^2 + \Vert \mathbf{r} \Vert_2^2$. Among all minimizers, $\hat{\mathbf{x}}_{\min} = A^{+}\mathbf{b}$ is the unique one lying in $\mathcal{R}(A^{\top}) = \mathcal{N}(A)^{\perp}$, and it has the smallest 2-norm.

**Theorem 4 (QR solves least squares).** Let $A = QR$ be a thin QR factorization of a full-column-rank $A$, and extend $Q$ to an orthogonal $\tilde{Q} = [\,Q \ \ Q_{\perp}\,] \in \mathbb{R}^{m \times m}$. Then

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert R\mathbf{x} - Q^{\top}\mathbf{b} \Vert_2^2 + \Vert Q_{\perp}^{\top}\mathbf{b} \Vert_2^2 ,
$$

so the minimizer solves the triangular system $R\hat{\mathbf{x}} = Q^{\top}\mathbf{b}$ by back substitution, and $\rho = \Vert Q_{\perp}^{\top}\mathbf{b} \Vert_2$.

**Theorem 5 (Condition-number squaring).** If $A$ has full column rank then $\kappa_2(A^{\top}A) = \kappa_2(A)^2$. Consequently solving the normal equations by Cholesky yields a relative error $\approx \varepsilon_{\text{mach}}\,\kappa_2(A)^2$, whereas Householder QR yields $\approx \varepsilon_{\text{mach}}\bigl(\kappa_2(A) + \kappa_2(A)^2\tan\theta\bigr)$ where $\sin\theta = \rho / \Vert \mathbf{b} \Vert_2$ — identical when the fit is poor, dramatically better when the fit is good.

**Theorem 6 (Backward stability of Householder QR; Higham).** The computed solution $\tilde{\mathbf{x}}$ from Householder QR is the exact least-squares solution of a nearby problem: there exist $\Delta A$, $\Delta \mathbf{b}$ with

$$
\frac{\Vert \Delta A \Vert_2}{\Vert A \Vert_2} \le c(m,n)\,\varepsilon_{\text{mach}}, \qquad \frac{\Vert \Delta \mathbf{b} \Vert_2}{\Vert \mathbf{b} \Vert_2} \le c(m,n)\,\varepsilon_{\text{mach}},
$$

such that $\tilde{\mathbf{x}}$ exactly minimizes $\Vert (A + \Delta A)\mathbf{x} - (\mathbf{b} + \Delta \mathbf{b}) \Vert_2$. Classical Gram–Schmidt has no such guarantee: its loss of orthogonality is proportional to $\kappa_2(A)^2$, while modified Gram–Schmidt loses orthogonality only proportionally to $\kappa_2(A)$.

**Theorem 7 (Ridge as SVD filtering).** With $A = U\Sigma V^{\top}$ of rank $r$ and $\lambda \gt 0$,

$$
\hat{\mathbf{x}}_\lambda = \sum_{i=1}^{r} \frac{\sigma_i^2}{\sigma_i^2 + \lambda} \cdot \frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\,\mathbf{v}_i ,
$$

so ridge multiplies each SVD coefficient by the *filter factor* $f_i = \sigma_i^2/(\sigma_i^2 + \lambda) \in (0,1)$. Truncated SVD is the hard-threshold analogue $f_i \in \{0, 1\}$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — The normal equations from orthogonal projection

**Claim.** (Theorem 1.) $\hat{\mathbf{x}}$ minimizes $\Vert A\mathbf{x} - \mathbf{b} \Vert_2$ if and only if $A^{\top}(\mathbf{b} - A\hat{\mathbf{x}}) = \mathbf{0}$.

**Proof.** Write an arbitrary $\mathbf{x} = \hat{\mathbf{x}} + \mathbf{d}$ and expand the squared residual:

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert A\hat{\mathbf{x}} - \mathbf{b} + A\mathbf{d} \Vert_2^2 = \Vert \mathbf{r} \Vert_2^2 - 2\mathbf{d}^{\top}A^{\top}\mathbf{r} + \Vert A\mathbf{d} \Vert_2^2 ,
$$

where $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$ and we used $\langle A\mathbf{d}, A\hat{\mathbf{x}} - \mathbf{b}\rangle = -\mathbf{d}^{\top}A^{\top}\mathbf{r}$.

*($\Leftarrow$) Sufficiency.* If $A^{\top}\mathbf{r} = \mathbf{0}$, the cross term vanishes identically and

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert \mathbf{r} \Vert_2^2 + \Vert A\mathbf{d} \Vert_2^2 \ge \Vert \mathbf{r} \Vert_2^2
$$

for every $\mathbf{d}$ — this is the Pythagorean identity of Theorem 3, and $\hat{\mathbf{x}}$ is a global minimizer.

*($\Rightarrow$) Necessity.* Suppose $\mathbf{g} := A^{\top}\mathbf{r} \neq \mathbf{0}$. Take $\mathbf{d} = t\mathbf{g}$ with $t \gt 0$ small. Then

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 - \Vert \mathbf{r} \Vert_2^2 = -2t \Vert \mathbf{g} \Vert_2^2 + t^2 \Vert A\mathbf{g} \Vert_2^2 ,
$$

which is strictly negative for all $0 \lt t \lt 2\Vert \mathbf{g} \Vert_2^2 / \Vert A\mathbf{g} \Vert_2^2$ (the interval is nonempty since $\mathbf{g} \neq \mathbf{0}$; if $A\mathbf{g} = \mathbf{0}$ then $\Vert \mathbf{g} \Vert_2^2 = \mathbf{g}^{\top}A^{\top}\mathbf{r} = (A\mathbf{g})^{\top}\mathbf{r} = 0$, a contradiction). So $\hat{\mathbf{x}}$ is not a minimizer. $\blacksquare$

Geometrically: the minimizing $A\hat{\mathbf{x}}$ is the foot of the perpendicular from $\mathbf{b}$ to the plane $\mathcal{R}(A)$, and $\mathbf{r}$ is the perpendicular itself. No calculus was required — only the parallelogram algebra of inner products.

### Proof 2 — The same equations from calculus, plus uniqueness

**Claim.** (Theorem 2.) $\phi(\mathbf{x}) = \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2$ is convex with $\nabla \phi = 2A^{\top}(A\mathbf{x} - \mathbf{b})$, and is *strictly* convex — hence has a unique minimizer — exactly when $\operatorname{rank}(A) = n$.

**Proof.** Expand:

$$
\phi(\mathbf{x}) = \mathbf{x}^{\top}A^{\top}A\mathbf{x} - 2\mathbf{b}^{\top}A\mathbf{x} + \mathbf{b}^{\top}\mathbf{b} .
$$

Differentiating the quadratic form gives $\nabla \phi(\mathbf{x}) = 2A^{\top}A\mathbf{x} - 2A^{\top}\mathbf{b}$ and $\nabla^2 \phi = 2A^{\top}A$. For any $\mathbf{z}$,

$$
\mathbf{z}^{\top}A^{\top}A\mathbf{z} = \Vert A\mathbf{z} \Vert_2^2 \ge 0,
$$

so $\nabla^2\phi \succeq 0$: $\phi$ is convex and every stationary point is a global minimum, giving the normal equations again.

Equality $\Vert A\mathbf{z} \Vert_2 = 0$ holds for some $\mathbf{z} \neq \mathbf{0}$ iff $\mathcal{N}(A) \neq \{\mathbf{0}\}$ iff $\operatorname{rank}(A) \lt n$. Hence $A^{\top}A \succ 0$ (positive definite, therefore invertible, therefore a unique solution) iff $A$ has full column rank. When $\operatorname{rank}(A) = r \lt n$, adding any $\mathbf{z} \in \mathcal{N}(A)$ to a minimizer leaves $A\mathbf{x}$ and hence $\phi$ unchanged, so the solution set is exactly $\hat{\mathbf{x}} + \mathcal{N}(A)$, an affine subspace of dimension $n - r$. $\blacksquare$

**Consistency of the normal equations.** They are *never* inconsistent: $A^{\top}\mathbf{b} \in \mathcal{R}(A^{\top}) = \mathcal{R}(A^{\top}A)$ because $\mathcal{N}(A^{\top}A) = \mathcal{N}(A)$ (if $A^{\top}A\mathbf{z} = \mathbf{0}$ then $\Vert A\mathbf{z} \Vert_2^2 = \mathbf{z}^{\top}A^{\top}A\mathbf{z} = 0$), and orthogonal complements of equal null spaces are equal ranges.

### Proof 3 — Forming $A^{\top}A$ squares the condition number

**Claim.** (Theorem 5.) For $A$ of full column rank, $\kappa_2(A^{\top}A) = \kappa_2(A)^2$.

**Proof.** Let $A = U\Sigma V^{\top}$ be the SVD, with $\Sigma = \operatorname{diag}(\sigma_1, \ldots, \sigma_n)$, $\sigma_1 \ge \cdots \ge \sigma_n \gt 0$. Then

$$
A^{\top}A = V\Sigma^{\top}U^{\top}U\Sigma V^{\top} = V\Sigma^2 V^{\top} ,
$$

which is a spectral (eigen) decomposition of the symmetric positive definite matrix $A^{\top}A$ with eigenvalues $\sigma_i^2$. For a symmetric positive definite matrix the 2-norm condition number is the ratio of largest to smallest eigenvalue, so

$$
\kappa_2(A^{\top}A) = \frac{\sigma_1^2}{\sigma_n^2} = \left(\frac{\sigma_1}{\sigma_n}\right)^{2} = \kappa_2(A)^2 . \qquad \blacksquare
$$

**Numerical consequence.** A backward-stable solve of a linear system with condition number $\kappa$ returns a solution with relative error about $\varepsilon_{\text{mach}}\kappa$. In IEEE double precision $\varepsilon_{\text{mach}} \approx 1.1 \times 10^{-16}$, so:

- $\kappa_2(A) = 10^{4}$: QR gives $\approx 10^{-12}$ relative error; normal equations give $\approx 10^{-8}$.
- $\kappa_2(A) = 10^{8}$: QR gives $\approx 10^{-8}$; the normal equations give $\approx 1$ — **no correct digits at all**, and Cholesky may even fail because the rounded $A^{\top}A$ is not numerically positive definite.

**The Läuchli example** makes the failure concrete. With $\varepsilon = 10^{-8} \lt \sqrt{\varepsilon_{\text{mach}}}$ take

$$
A = \begin{bmatrix} 1 & 1 \\ \varepsilon & 0 \\ 0 & \varepsilon \end{bmatrix}, \qquad A^{\top}A = \begin{bmatrix} 1 + \varepsilon^2 & 1 \\ 1 & 1 + \varepsilon^2 \end{bmatrix} .
$$

Here $\operatorname{rank}(A) = 2$ exactly and $\kappa_2(A) = \sqrt{2}/\varepsilon \approx 1.4 \times 10^{8}$. But $1 + \varepsilon^2 = 1 + 10^{-16}$ rounds to $1$ in double precision, so the *computed* $A^{\top}A$ is the singular matrix of all ones: the information distinguishing the two columns has been destroyed by the multiplication itself, before any solve begins. Householder QR never forms this product and returns full accuracy.

### Proof 4 — QR reduces least squares to back substitution

**Claim.** (Theorem 4.) With $A = QR$ thin and $\tilde{Q} = [\,Q \ \ Q_\perp\,]$ orthogonal, $\hat{\mathbf{x}}$ solves $R\mathbf{x} = Q^{\top}\mathbf{b}$ and $\rho = \Vert Q_\perp^{\top}\mathbf{b} \Vert_2$.

**Proof.** Orthogonal matrices are 2-norm isometries: $\Vert \tilde{Q}^{\top}\mathbf{y} \Vert_2^2 = \mathbf{y}^{\top}\tilde{Q}\tilde{Q}^{\top}\mathbf{y} = \Vert \mathbf{y} \Vert_2^2$. Apply $\tilde{Q}^{\top}$ to the residual — legal precisely because it changes nothing we are measuring:

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert \tilde{Q}^{\top}(A\mathbf{x} - \mathbf{b}) \Vert_2^2 = \left\Vert \begin{bmatrix} Q^{\top}A\mathbf{x} - Q^{\top}\mathbf{b} \\ Q_\perp^{\top}A\mathbf{x} - Q_\perp^{\top}\mathbf{b} \end{bmatrix} \right\Vert_2^2 .
$$

Now $Q^{\top}A = Q^{\top}QR = R$ and $Q_\perp^{\top}A = Q_\perp^{\top}QR = 0$ because the columns of $Q_\perp$ are orthogonal to those of $Q$. Hence

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert R\mathbf{x} - Q^{\top}\mathbf{b} \Vert_2^2 + \Vert Q_\perp^{\top}\mathbf{b} \Vert_2^2 .
$$

The second term is independent of $\mathbf{x}$ — it is the unavoidable part of $\mathbf{b}$ lying outside $\mathcal{R}(A)$. The first term is nonnegative and can be driven to exactly zero, because $R$ is nonsingular when $\operatorname{rank}(A) = n$: solve $R\hat{\mathbf{x}} = Q^{\top}\mathbf{b}$ by back substitution in $O(n^2)$ flops. Therefore $\hat{\mathbf{x}}$ is the minimizer and $\rho = \Vert Q_\perp^{\top}\mathbf{b} \Vert_2$. $\blacksquare$

**Why this is better than the normal equations.** Substituting $A = QR$ into $A^{\top}A\mathbf{x} = A^{\top}\mathbf{b}$ gives $R^{\top}R\mathbf{x} = R^{\top}Q^{\top}\mathbf{b}$, i.e. $R^{\top}$ cancels and the two approaches agree *in exact arithmetic* — the Cholesky factor of $A^{\top}A$ is $R$ itself. In floating point they differ completely, because QR obtains $R$ from $A$ directly (condition number $\kappa_2(A)$), while Cholesky obtains it from the already-degraded $A^{\top}A$ (condition number $\kappa_2(A)^2$).

### Proof 5 — The pseudoinverse gives the minimum-norm solution

**Claim.** (Theorem 3.) If $\operatorname{rank}(A) = r$, then $\hat{\mathbf{x}}_{\min} = A^{+}\mathbf{b}$ minimizes $\Vert A\mathbf{x} - \mathbf{b} \Vert_2$, and among all such minimizers it uniquely minimizes $\Vert \mathbf{x} \Vert_2$.

**Proof.** Use the full SVD $A = U\Sigma V^{\top}$ and change variables $\mathbf{y} = V^{\top}\mathbf{x}$, $\mathbf{c} = U^{\top}\mathbf{b}$. Since $U$ and $V$ are orthogonal, both norms are preserved: $\Vert \mathbf{x} \Vert_2 = \Vert \mathbf{y} \Vert_2$ and

$$
\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert U(\Sigma \mathbf{y} - \mathbf{c}) \Vert_2^2 = \sum_{i=1}^{r} (\sigma_i y_i - c_i)^2 + \sum_{i=r+1}^{m} c_i^2 .
$$

The objective is now separable. The first sum is minimized (to zero) by $y_i = c_i/\sigma_i$ for $i \le r$; the second is a constant, giving $\rho^2 = \sum_{i \gt r} c_i^2$. The components $y_{r+1}, \ldots, y_n$ do **not appear** — they are free, which is exactly the null-space freedom of Theorem 2.

Among all minimizers, $\Vert \mathbf{x} \Vert_2^2 = \Vert \mathbf{y} \Vert_2^2 = \sum_{i \le r} (c_i/\sigma_i)^2 + \sum_{i \gt r} y_i^2$ is minimized uniquely by setting every free component to zero. Undoing the change of variables,

$$
\hat{\mathbf{x}}_{\min} = V\mathbf{y} = \sum_{i=1}^{r} \frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\,\mathbf{v}_i = V_r\Sigma_r^{-1}U_r^{\top}\mathbf{b} = A^{+}\mathbf{b} . \qquad \blacksquare
$$

Two corollaries fall out immediately. (i) $\hat{\mathbf{x}}_{\min} \in \operatorname{span}\{\mathbf{v}_1, \ldots, \mathbf{v}_r\} = \mathcal{R}(A^{\top}) = \mathcal{N}(A)^{\perp}$. (ii) The expansion exposes the danger: any noise component along $\mathbf{u}_i$ is amplified by $1/\sigma_i$, so small singular values, not large ones, destroy ill-posed fits. That single observation is the whole motivation for regularization.

### Proof 6 — Tikhonov regularization: augmented system and filter factors

**Claim.** (Theorem 7.) The ridge problem $\min_{\mathbf{x}} \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 + \lambda \Vert \mathbf{x} \Vert_2^2$ has the unique solution $\hat{\mathbf{x}}_\lambda = (A^{\top}A + \lambda I)^{-1}A^{\top}\mathbf{b}$ for every $\lambda \gt 0$, equals an ordinary least-squares problem for an augmented matrix, and acts on the SVD as multiplication by $f_i = \sigma_i^2/(\sigma_i^2 + \lambda)$.

**Proof.** *Augmented form.* Define the stacked matrix and vector

$$
A_\lambda = \begin{bmatrix} A \\ \sqrt{\lambda}\,I_n \end{bmatrix} \in \mathbb{R}^{(m+n) \times n}, \qquad \mathbf{b}_\lambda = \begin{bmatrix} \mathbf{b} \\ \mathbf{0} \end{bmatrix} \in \mathbb{R}^{m+n} .
$$

Then $\Vert A_\lambda \mathbf{x} - \mathbf{b}_\lambda \Vert_2^2 = \Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 + \lambda\Vert \mathbf{x} \Vert_2^2$, so the ridge problem is *literally* a least-squares problem — and can therefore be solved by QR on $A_\lambda$, never forming any cross product. Since $A_\lambda^{\top}A_\lambda = A^{\top}A + \lambda I \succ 0$ for $\lambda \gt 0$, $A_\lambda$ always has full column rank and the solution is unique even when $A$ is rank deficient; the normal equations of the augmented problem are $(A^{\top}A + \lambda I)\mathbf{x} = A^{\top}\mathbf{b}$.

*Filter factors.* Insert $A = U\Sigma V^{\top}$:

$$
\hat{\mathbf{x}}_\lambda = (V\Sigma^{\top}\Sigma V^{\top} + \lambda V V^{\top})^{-1} V\Sigma^{\top}U^{\top}\mathbf{b} = V(\Sigma^{\top}\Sigma + \lambda I)^{-1}\Sigma^{\top}U^{\top}\mathbf{b} .
$$

The middle factor is diagonal with entries $\sigma_i/(\sigma_i^2 + \lambda)$, so

$$
\hat{\mathbf{x}}_\lambda = \sum_{i=1}^{r} \frac{\sigma_i}{\sigma_i^2 + \lambda}\,(\mathbf{u}_i^{\top}\mathbf{b})\,\mathbf{v}_i = \sum_{i=1}^{r} \underbrace{\frac{\sigma_i^2}{\sigma_i^2 + \lambda}}_{f_i} \cdot \frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\,\mathbf{v}_i . \qquad \blacksquare
$$

**Interpretation.** When $\sigma_i \gg \sqrt{\lambda}$, $f_i \approx 1$: well-determined directions pass through untouched. When $\sigma_i \ll \sqrt{\lambda}$, $f_i \approx \sigma_i^2/\lambda \approx 0$: the noise-amplifying directions are suppressed, and the dangerous factor $1/\sigma_i$ is replaced by $\sigma_i/\lambda$, which is *small*. The condition number of the regularized problem satisfies $\kappa_2(A_\lambda) = \sqrt{(\sigma_1^2 + \lambda)/(\sigma_n^2 + \lambda)} \le \sqrt{(\sigma_1^2 + \lambda)/\lambda}$ — bounded no matter how ill-conditioned $A$ was. Truncated SVD is the same idea with the hard filter $f_i = 1$ for $\sigma_i \gt \tau$ and $f_i = 0$ otherwise; as $\lambda \to 0^{+}$, $\hat{\mathbf{x}}_\lambda \to A^{+}\mathbf{b}$, recovering the minimum-norm solution.

## 4. Computational & Algorithmic Insights

### Householder QR: the production algorithm

Householder QR builds $R$ by applying $n$ reflectors, each zeroing everything below the diagonal in one column:

$$
H_n \cdots H_2 H_1 A = \begin{bmatrix} R \\ 0 \end{bmatrix}, \qquad H_k = I - 2\frac{\mathbf{v}_k\mathbf{v}_k^{\top}}{\mathbf{v}_k^{\top}\mathbf{v}_k} .
$$

Three implementation points decide whether it works.

1. **Sign choice.** Take $\alpha = -\operatorname{sign}(a_1)\Vert \mathbf{a} \Vert_2$ so that $\mathbf{v} = \mathbf{a} - \alpha\mathbf{e}_1$ has a *large* first component. The opposite sign risks cancellation when $\mathbf{a}$ is already nearly a multiple of $\mathbf{e}_1$.
2. **Never form $H_k$.** Apply it as a rank-one update, $H\mathbf{x} = \mathbf{x} - 2\frac{\mathbf{v}^{\top}\mathbf{x}}{\mathbf{v}^{\top}\mathbf{v}}\mathbf{v}$, which costs $O(m)$ instead of $O(m^2)$.
3. **Never form $Q$.** Store the $\mathbf{v}_k$ in the annihilated lower triangle and apply the reflectors in sequence to $\mathbf{b}$ to obtain $Q^{\top}\mathbf{b}$ directly.

Cost: $2mn^2 - \tfrac{2}{3}n^3$ flops, versus $mn^2 + \tfrac{1}{3}n^3$ for normal equations plus Cholesky. QR is about twice the work and buys back half your digits — an overwhelmingly good trade.

### Givens rotations and the Gram–Schmidt family

**Givens rotations** zero one entry at a time using a $2 \times 2$ plane rotation

$$
G = \begin{bmatrix} c & s \\ -s & c \end{bmatrix}, \qquad c = \frac{a}{\sqrt{a^2+b^2}}, \quad s = \frac{b}{\sqrt{a^2+b^2}} .
$$

They cost more than Householder for a dense matrix but are the method of choice when $A$ is sparse or already nearly triangular (Hessenberg reduction in eigenvalue algorithms), and when a factorization must be *updated* after adding a row of data — the recursive-least-squares update used in adaptive filters and Kalman-style estimators.

**Classical Gram–Schmidt (CGS)** orthogonalizes column $\mathbf{a}_k$ against all previous $\mathbf{q}_j$ using the *original* vector; **modified Gram–Schmidt (MGS)** subtracts each projection immediately and orthogonalizes the *running remainder*. Algebraically identical, numerically not:

$$
\Vert I - Q^{\top}Q \Vert_2 \approx \begin{cases} c\,\varepsilon_{\text{mach}}\,\kappa_2(A)^2 & \text{(CGS)} \\ c\,\varepsilon_{\text{mach}}\,\kappa_2(A) & \text{(MGS)} \\ c\,\varepsilon_{\text{mach}} & \text{(Householder)} \end{cases}
$$

CGS with reorthogonalization ("twice is enough", Kahan–Parlett) restores full orthogonality at double cost. Use Householder for dense factorizations; MGS when the columns of $Q$ must appear one at a time, as in Arnoldi/GMRES.

### Method selection

| Method | Cost (flops) | Accuracy | Use when |
| :--- | :--- | :--- | :--- |
| Normal equations + Cholesky | $mn^2 + \tfrac{1}{3}n^3$ | error $\sim \varepsilon\,\kappa_2(A)^2$ | $\kappa_2(A)$ small, $m \gg n$, speed critical |
| Householder QR | $2mn^2 - \tfrac{2}{3}n^3$ | backward stable | Default for dense full-rank problems |
| QR with column pivoting | $\approx 2mn^2$ plus pivot search | reveals numerical rank | Rank in doubt, subset selection |
| SVD | $\approx 2mn^2 + 11n^3$ | most reliable | Rank deficient, ill-posed, need filter factors |
| Ridge via augmented QR | QR on an $(m{+}n) \times n$ matrix | stable, bounded $\kappa$ | Regularized/ill-posed problems |
| CG / LSQR (iterative) | $O(mn)$ per iteration | depends on $\kappa_2(A)$ | $n$ huge, $A$ sparse or matrix-free |

**Rank determination** is a numerical, not algebraic, decision. The standard rule declares numerical rank $r$ = number of singular values exceeding $\tau = \max(m,n)\,\sigma_1\,\varepsilon_{\text{mach}}$ (the default in `numpy.linalg.matrix_rank` and `lstsq`). QR with column pivoting, $AP = QR$ with $\vert r_{11} \vert \ge \vert r_{22} \vert \ge \cdots$, gives a cheaper rank estimate and simultaneously selects a well-conditioned subset of columns.

### Weighted least squares and the conditioning of polynomial fits

**Weighted least squares.** If the measurement errors are independent with variances $\sigma_i^2$, maximum likelihood weights each residual by $1/\sigma_i$:

$$
\min_{\mathbf{x}} \sum_{i=1}^{m} w_i \bigl(\mathbf{a}_i^{\top}\mathbf{x} - b_i\bigr)^2 = \min_{\mathbf{x}} \Vert W^{1/2}(A\mathbf{x} - \mathbf{b}) \Vert_2^2, \qquad w_i = 1/\sigma_i^2 ,
$$

with normal equations $A^{\top}WA\,\hat{\mathbf{x}} = A^{\top}W\mathbf{b}$. Computationally, scale the rows of $A$ and $\mathbf{b}$ by $\sqrt{w_i}$ and run ordinary QR. For correlated noise with covariance $C$, use the Cholesky factor $C = LL^{\top}$ and solve the whitened problem in $L^{-1}A$, $L^{-1}\mathbf{b}$ — this is generalized least squares. Beware: extreme weights inflate $\kappa_2(W^{1/2}A)$, so weighting is not free.

**Polynomial fitting.** Fitting $p(t) = \sum_{j=0}^{d} x_j t^j$ produces the Vandermonde matrix $A_{ij} = t_i^{\,j}$, whose columns $1, t, t^2, \ldots$ become nearly parallel as $d$ grows. On $50$ equispaced nodes in $[0,1]$ the 2-norm condition number is roughly $6\times 10^{2}$ at degree 4, $4\times 10^{6}$ at degree 9, and $2\times 10^{10}$ at degree 14 — so a degree-14 monomial fit in double precision has no reliable coefficients at all, even though the *fitted curve* may still be accurate. The fix is a change of basis: shift and scale the nodes to $[-1,1]$ and use Chebyshev or Legendre polynomials, which are near-orthogonal under the discrete inner product and keep $\kappa_2$ at modest values. This is the same lesson as Topic 04's interpolation nodes, arriving through conditioning rather than through the Lebesgue constant.

## 5. Real-World Physics & AI/ML Applications

**Geodesy and GPS.** A receiver measures pseudoranges to $m \ge 5$ satellites to determine 3 position coordinates plus a clock bias. After linearizing about a nominal position, each satellite contributes one row, and the correction is a least-squares solve. The *geometric dilution of precision* (GDOP) quoted by every receiver is nothing but $\sqrt{\operatorname{tr}\bigl[(A^{\top}A)^{-1}\bigr]}$ — a direct statement about the conditioning of the design matrix. Satellites clustered in one part of the sky make $A$'s columns nearly parallel, $\sigma_n$ tiny, and the fix unreliable, no matter how precise the timing.

**Orbit determination and the birth of the method.** Gauss recovered the asteroid Ceres in 1801 by fitting an orbit to a handful of noisy sightings; Legendre published the method in 1805. The problem is the archetype: far more observations than the six orbital elements, each observation corrupted by atmospheric and instrumental noise.

**Experimental physics.** Calibration curves, spectral line fitting, and the extraction of a decay constant from counts all reduce to weighted least squares, with $w_i = 1/\sigma_i^2$ from Poisson counting statistics. Reporting a fitted parameter without its covariance $\sigma^2 (A^{\top}A)^{-1}$ is incomplete — and computing that covariance stably means using $R^{-1}R^{-\top}$ from the QR factorization, not inverting $A^{\top}A$.

**Structural and inverse problems.** Tomography, deconvolution, and gravimetry produce matrices whose singular values decay to zero (discrete ill-posed problems). Here the unregularized solution is pure amplified noise, and the practical question is choosing $\lambda$ — by the discrepancy principle ($\rho(\lambda) \approx$ known noise level), generalized cross-validation, or the corner of Hansen's L-curve.

### Machine learning

- **Linear regression.** The ordinary least-squares estimator $\hat{\boldsymbol{\beta}} = (X^{\top}X)^{-1}X^{\top}\mathbf{y}$ is the same object under a different name; the Gauss–Markov theorem says it is the best linear unbiased estimator, and under Gaussian noise it coincides with the MLE. Libraries such as scikit-learn's `LinearRegression` call LAPACK's `gelsd` (SVD-based), not the textbook formula.
- **Ridge regression and multicollinearity.** Correlated features make columns of $X$ nearly dependent, $\sigma_n \to 0$, and coefficients explode with alternating signs while the fit itself looks fine. Ridge is Tikhonov regularization: it shrinks along small-$\sigma$ directions (Proof 6), yields the MAP estimate under a Gaussian prior $\boldsymbol{\beta} \sim \mathcal{N}(\mathbf{0}, \tau^2 I)$ with $\lambda = \sigma^2/\tau^2$, and trades a small bias for a large variance reduction. The effective number of parameters is $\operatorname{tr}\bigl[X(X^{\top}X + \lambda I)^{-1}X^{\top}\bigr] = \sum_i f_i$ — the sum of the filter factors.
- **Closed form versus gradient descent.** The normal equations cost $O(mn^2)$ and are exact; gradient descent costs $O(mn)$ per step and converges linearly with contraction factor $(\kappa - 1)/(\kappa + 1)$ where $\kappa = \kappa_2(X^{\top}X) = \kappa_2(X)^2$. Feature standardization is therefore not cosmetic — it directly shrinks $\kappa$ and accelerates training. For $n$ beyond a few thousand, or for streaming data, iterative methods (LSQR, conjugate gradients on the normal equations, SGD) win; LSQR is preferred over naive CG because it works with $X$ rather than $X^{\top}X$ and so converges at a rate governed by $\kappa_2(X)$.
- **Beyond linear models.** The last layer of a deep network trained with squared loss and frozen features is exactly a least-squares problem; Gauss–Newton and Levenberg–Marquardt solve a *sequence* of linear least-squares problems (with $\lambda$ playing the trust-region role); kernel ridge regression, Gaussian-process regression, extreme learning machines, random-feature models, and the linear probes used to interpret representation quality are all direct applications. The pseudoinverse also explains the *minimum-norm interpolation* regime central to modern double-descent theory: when $n \gt m$, gradient descent from zero converges to $X^{+}\mathbf{y}$, the minimum-norm interpolant of Proof 5.

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Normal equations | $\hat{\mathbf{x}}$ minimizes iff $A^{\top}A\hat{\mathbf{x}} = A^{\top}\mathbf{b}$ iff $\mathbf{r} \perp \mathcal{R}(A)$ |
| Uniqueness | Unique iff $\operatorname{rank}(A) = n$; otherwise solution set is $\hat{\mathbf{x}} + \mathcal{N}(A)$ |
| Pythagoras | $\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 = \Vert A(\mathbf{x} - \hat{\mathbf{x}}) \Vert_2^2 + \rho^2$ |
| QR solution | $R\hat{\mathbf{x}} = Q^{\top}\mathbf{b}$, residual $\rho = \Vert Q_\perp^{\top}\mathbf{b} \Vert_2$ |
| Condition squaring | $\kappa_2(A^{\top}A) = \kappa_2(A)^2$; QR error $\sim \varepsilon\kappa_2(A)$, Cholesky error $\sim \varepsilon\kappa_2(A)^2$ |
| Pseudoinverse | $\hat{\mathbf{x}}_{\min} = A^{+}\mathbf{b} = \sum_{i \le r} \sigma_i^{-1}(\mathbf{u}_i^{\top}\mathbf{b})\mathbf{v}_i$, minimal $\Vert \mathbf{x} \Vert_2$ |
| Ridge filter | $f_i = \sigma_i^2/(\sigma_i^2 + \lambda)$; augmented matrix stacks $A$ over $\sqrt{\lambda}I$ |
| Orthogonality loss | CGS $\sim \varepsilon\kappa^2$, MGS $\sim \varepsilon\kappa$, Householder $\sim \varepsilon$ |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Normal equations, projection geometry | Trefethen & Bau, *Numerical Linear Algebra* | Lectures 6, 11 |
| Gram–Schmidt, CGS vs MGS stability | Trefethen & Bau | Lectures 7–8 |
| Householder and Givens QR | Golub & Van Loan, *Matrix Computations* | Ch. 5.1–5.2 |
| Least squares by QR, SVD, pivoting | Golub & Van Loan | Ch. 5.3–5.5 |
| Comprehensive least-squares theory | Björck, *Numerical Methods for Least Squares Problems* | Chs. 1–4 |
| Conditioning and perturbation of LS | Trefethen & Bau; Higham | Lecture 18; Ch. 20 |
| Läuchli example, method comparison | Heath, *Scientific Computing* | Ch. 3 |
| Discrete LS and orthogonal polynomials | Burden & Faires, *Numerical Analysis* | Ch. 8.1–8.2 |
| Tikhonov, TSVD, L-curve, filter factors | Hansen, *Rank-Deficient and Discrete Ill-Posed Problems* | Chs. 3–7 |
| Ridge, lasso, bias–variance | Hastie, Tibshirani & Friedman, *ESL* | Ch. 3 |
| Applied data fitting, classification | Boyd & Vandenberghe, *Introduction to Applied Linear Algebra* | Chs. 12–15 |
| LSQR and iterative least squares | Paige & Saunders (1982), *ACM TOMS* 8(1) | Algorithm 583 |

**Primary references.** Björck (1996); Golub & Van Loan (Ch. 5); Trefethen & Bau (Lectures 6–11, 18); Heath (Ch. 3); Higham (Chs. 19–20); Hansen (1998); Hastie et al. (Ch. 3).